# E4 · Uhura — les six états de modèle sur l'axe **Honest**

C'est le notebook qui porte le claim. Il évalue **six états** sur les mêmes 808 questions,
et c'est leur comparaison deux à deux qui répond aux questions du projet.

| état | modèle | adaptateur | ce qu'il sert à mesurer |
| :---- | :---- | :---- | :---- |
| **A0** | Qwen3.5-4B-Base | — | référence |
| **A1** | AfriqueQwen-50Langs | — | écart **avant** tout alignement |
| **A2s** | Qwen3.5-4B-Base | SFT | contribution du SFT seul |
| **A3s** | AfriqueQwen-50Langs | SFT | contribution du SFT seul |
| **A2d** | Qwen3.5-4B-Base | SFT + DPO | **le claim** |
| **A3d** | AfriqueQwen-50Langs | SFT + DPO | **le claim** |

## Les trois écarts, et pourquoi il faut les trois

```
A1  - A0    ecart de depart        deja mesure: +0,021, p = 0,21 -- rien
A3s - A2s   apres SFT seul         JAMAIS MESURE
A3d - A2d   apres SFT + DPO        L'OBJECTIF
```

Sans le point intermédiaire, un écart A3d − A2d positif ne dirait pas **quelle étape** l'a
produit. C'est exactement l'ablation qu'InstructGPT fait entre ses deux étapes.

## Pourquoi cette mesure et pas les marges de récompense

Les marges du DPO ne sont **pas comparables entre bras** : la récompense implicite vaut
`β · log(π / π_ref)`, où `π_ref` est le modèle gelé **propre à chaque bras**. Une marge plus
grande dit qu'un bras s'est davantage éloigné de *son* point de départ, pas qu'il est mieux
aligné. Ce sont en outre des métriques d'entraînement.

**Ici, les six états répondent aux mêmes questions, jamais vues à l'entraînement.** C'est la
seule échelle commune.

⚠️ Le contenu d'Uhura est **occidental**, professionnellement traduit. On mesure la véracité
sur du savoir occidental exprimé en haoussa, pas sur du savoir africain. À déclarer.

### Réglages Kaggle
Accelerator **T4 x2**, internet activé, datasets `afrique-safety-dpo-code` et
`afrique-safety-dpo-adapters` attachés. Durée attendue : **~2 h** (6 × 20 min).

## 0 · Environnement

Même préparation que partout ailleurs, et pour les mêmes raisons mesurées : invalidation du
cache `bitsandbytes` de `transformers`, et `expandable_segments` contre la fragmentation que
provoque un vocabulaire de 248 077 tokens.

In [ ]:
!pip install -q -U "transformers==5.16.1" "peft==0.20.0" bitsandbytes accelerate datasets

In [ ]:
import os, sys
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


def racine_du_code():
    """Le dossier contenant src/paths.py, ou qu'il soit monte.

    Kaggle ne monte pas tous les datasets a la meme profondeur -- un kernel a montre
    /kaggle/input/<slug>/src/..., un autre /kaggle/input/datasets/<user>/<slug>/src/...
    Un chemin code en dur leve StopIteration sans dire ce qui manque.
    """
    for base in (Path("/kaggle/input"), Path.cwd(), *Path.cwd().parents):
        if not base.exists():
            continue
        for trouve in base.rglob("paths.py"):
            if trouve.parent.name == "src":
                return trouve.parents[1]
    raise RuntimeError("src/paths.py introuvable. Attacher afrique-safety-dpo-code.")


def chemin_adaptateur(nom):
    """Le dossier nomme `nom` contenant un adapter_config.json.

    On cherche par NOM DE DOSSIER: Kaggle supprime le dossier de tete quand il est seul a
    la racine du zip, donc `adapters/A3_s42_sft/` arrive comme `A3_s42_sft/`.
    """
    for base in (Path("/kaggle/input"), Path.cwd(), *Path.cwd().parents):
        if not base.exists():
            continue
        for trouve in base.rglob("adapter_config.json"):
            if trouve.parent.name == nom:
                return trouve.parent
    raise RuntimeError(f"adaptateur {nom} introuvable. Attacher afrique-safety-dpo-adapters.")


RACINE_CODE = racine_du_code()
sys.path.insert(0, str(RACINE_CODE))

import importlib
import src.eval_mcq
importlib.reload(src.eval_mcq)
from src.eval_mcq import binomial_two_sided_p, evaluate_mcq, mcnemar_p

import torch, transformers, peft
print("code :", RACINE_CODE)
print(f"{torch.cuda.get_device_name(0)} | "
      f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} Go")
print(f"transformers {transformers.__version__} | peft {peft.__version__}")

## 1 · Les six états

Un état est un couple **(backbone, adaptateur)**. L'adaptateur `None` désigne le modèle brut.

**Le garde-fou de la cellule est essentiel.** Un adaptateur posé sur le mauvais backbone
produit du bruit **sans lever la moindre erreur** — c'est le mode d'échec le plus dangereux
de tout le dispositif. Chaque `adapter_config.json` déclare sa base, et on la compare à
celle qu'on s'apprête à charger.

In [ ]:
QWEN = "Qwen/Qwen3.5-4B-Base"
AFRIQUE = "McGill-NLP/AfriqueQwen3.5-4B-50Langs"

ETATS = [
    ("A0_base",       QWEN,    None),
    ("A1_base",       AFRIQUE, None),
    ("A2s_sft",       QWEN,    "A2_s42_sft"),
    ("A3s_sft",       AFRIQUE, "A3_s42_sft"),
    ("A2d_sft_dpo",   QWEN,    "A2_s42_dpo"),
    ("A3d_sft_dpo",   AFRIQUE, "A3_s42_dpo"),
]

import json

CHEMINS = {}
for nom, backbone, adaptateur in ETATS:
    if adaptateur is None:
        CHEMINS[nom] = None
        print(f"{nom:<14} {backbone}")
        continue
    chemin = chemin_adaptateur(adaptateur)
    declaree = json.loads((chemin / "adapter_config.json").read_text())["base_model_name_or_path"]
    assert declaree == backbone, f"{adaptateur} entraine sur {declaree}, pas sur {backbone}"
    CHEMINS[nom] = chemin
    print(f"{nom:<14} {backbone}  +  {chemin.name}")

print(f"\n{len(ETATS)} etats, toutes les bases declarees concordent")

## 2 · Le jeu — et l'exclusion des questions vues à l'entraînement

⚠️ **Uhura `ha_generation` et `ha_multiple_choice` sont le même TruthfulQA.** Mesuré :
**785 des 808 questions du QCM apparaissent aussi dans le jeu de génération**, soit 97,2 %.

Or le DPO s'entraîne précisément sur `ha_generation`. Évaluer les états A2d et A3d sur les
808 questions reviendrait donc à **mesurer la mémorisation** : 625 d'entre elles ont été vues
à l'entraînement. Les états bruts et SFT ne seraient pas avantagés de la même façon, et
l'écart mesuré serait un artefact.

La cellule **recalcule le découpage du DPO** avec la même graine — elle ne fait pas confiance
à une liste écrite ailleurs — et ne garde que les questions jamais vues :

| | |
| :---- | ---: |
| QCM total | 808 |
| vues au DPO, **exclues** | 625 *(77,4 %)* |
| **jeu propre** | **183** *(22,6 %)* |
| — dont partition d'évaluation du DPO | 160 |
| — dont absentes de `ha_generation` | 23 |

**Le prix à payer est la puissance statistique** : 183 questions au lieu de 808. Un petit
écart deviendra difficile à distinguer du bruit. C'est un coût réel, mais mesurer sur des
données d'entraînement ne serait pas un compromis — ce serait une erreur.

**E2 et E3 ne sont pas touchés.** AfriMGSM et AfriHate n'ont jamais servi à l'entraînement :
leurs 250 et 1 049 exemples gardent toute leur puissance. Seul l'axe sur lequel on entraîne
est concerné, ce qui est logique.

In [ ]:
import collections

from datasets import load_dataset

from src.data import (build_preference_pairs, build_uhura_pairs, filter_by_axis,
                      load_ubuntuguard_rows, split_by_base_stem)
from src.paths import resolve_roots

# Recalcule le decoupage du DPO plutot que de faire confiance a une liste ecrite ailleurs.
# Meme graine de partition, meme construction: c'est la seule facon de garantir que le jeu
# exclu ici est exactement celui qui a servi a l'entrainement.
GRAINE_SPLIT = 42
R = resolve_roots()
gen = load_dataset("masakhane/uhura-truthfulqa", "ha_generation", split="test")
paires = build_uhura_pairs(list(gen), "Hausa") + filter_by_axis(
    build_preference_pairs(
        [r for r in load_ubuntuguard_rows(R["ubuntuguard"]) if r["language"] == "Hausa"]
    ), "honest")
tr_dpo, _ = split_by_base_stem(paires, seed=GRAINE_SPLIT)
vues = {p["base_stem"] for p in tr_dpo}

uhura = load_dataset("masakhane/uhura-truthfulqa", "ha_multiple_choice", split="test")
toutes = list(uhura)
questions = [q for q in toutes if q["question"].strip() not in vues]

print(f"QCM total     : {len(toutes)}")
print(f"  vues au DPO : {len(toutes)-len(questions)}  ({(len(toutes)-len(questions))/len(toutes)*100:.1f} %)  EXCLUES")
print(f"  jeu propre  : {len(questions)}  ({len(questions)/len(toutes)*100:.1f} %)")
assert questions, "aucune question propre -- verifier la graine de partition"
assert not ({q["question"].strip() for q in questions} & vues), "contamination residuelle"
print("\ncontamination residuelle : 0")

plancher = sum(1 / len(l["mc1_targets"]["choices"]) for l in questions) / len(questions)
print(f"plancher aleatoire sur le jeu propre : {plancher:.4f}")
print("options par question :",
      dict(sorted(collections.Counter(
          len(l["mc1_targets"]["choices"]) for l in questions).items())))

## 3 · Évaluation

Chargement en **4 bits**, comme à l'entraînement : évaluer en pleine précision un modèle
entraîné en QLoRA mesurerait autre chose que ce qu'on aligne.

**La justesse question par question est conservée pour chaque état.** Sans elle, aucun test
apparié n'est possible — et c'est le test apparié qui tranche, puisque les six états
répondent aux *mêmes* questions.

Écriture **après chaque état** : une session coupée au cinquième garde les quatre premiers.

In [ ]:
import gc, time

from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

quant = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16,
)

SORTIES = Path("/kaggle/working/resultats")
SORTIES.mkdir(parents=True, exist_ok=True)

e4, justesse = {}, {}

for nom, backbone, adaptateur in ETATS:
    if nom in e4:                       # reprise apres interruption
        continue
    print(f"--- {nom} : chargement", flush=True)
    t0 = time.time()
    tok = AutoTokenizer.from_pretrained(backbone)
    modele = AutoModelForCausalLM.from_pretrained(
        backbone, quantization_config=quant, device_map={"": 0}, dtype=torch.float16
    )
    if CHEMINS[nom] is not None:
        modele = PeftModel.from_pretrained(modele, str(CHEMINS[nom]))
    modele = modele.eval()

    print(f"--- {nom} : {len(questions)} questions", flush=True)
    r = evaluate_mcq(modele, tok, questions)
    r["p_hasard"] = binomial_two_sided_p(r["correct"], r["n"], r["random_baseline"])
    r["min"] = round((time.time() - t0) / 60, 1)
    justesse[nom] = [q["gold"] == q["predicted"] for q in r.pop("per_question")]
    e4[nom] = r

    (SORTIES / "E4_uhura.json").write_text(
        json.dumps({"jeu": "uhura ha_multiple_choice", "n": len(questions),
                    "resultats": e4, "justesse": justesse},
                   indent=2, ensure_ascii=False), encoding="utf-8")

    print(f"{nom:<14} {r['accuracy']:.4f}  (hasard {r['random_baseline']:.4f}, "
          f"p={r['p_hasard']:.3g})  [{r['min']} min]", flush=True)

    del modele, tok
    gc.collect(); torch.cuda.empty_cache()

## 4 · Les trois écarts

Chacun est jugé par **McNemar**, le test apparié : les états répondent aux mêmes questions,
et les traiter comme des échantillons indépendants jetterait de la puissance statistique déjà
payée en GPU. Seuls les désaccords portent de l'information.

In [ ]:
import pandas as pd

tableau = pd.DataFrame(e4).T[["n", "correct", "accuracy", "random_baseline", "p_hasard", "min"]]
tableau.columns = ["n", "justes", "exactitude", "hasard", "p vs hasard", "min"]
display(tableau.round(4))

COMPARAISONS = [
    ("ecart de depart",   "A0_base",     "A1_base"),
    ("apres SFT seul",    "A2s_sft",     "A3s_sft"),
    ("apres SFT + DPO",   "A2d_sft_dpo", "A3d_sft_dpo"),
]

print("\n" + "=" * 66)
for etiquette, controle, cible in COMPARAISONS:
    if controle not in e4 or cible not in e4:
        print(f"{etiquette:<18} incomplet"); continue
    ecart = e4[cible]["accuracy"] - e4[controle]["accuracy"]
    mc = mcnemar_p(justesse[controle], justesse[cible])
    verdict = "ECART REEL" if mc["p"] < 0.05 else "indiscernable de zero"
    print(f"{etiquette:<18} {ecart:+.4f}   "
          f"{controle.split('_')[0]} seul juste {mc['only_first']:>3} | "
          f"{cible.split('_')[0]} seul juste {mc['only_second']:>3} | "
          f"desaccords {mc['discordant']:>3}")
    print(f"{'':<18} p = {mc['p']:.4g}  ->  {verdict}")

## 5 · Ce que l'alignement a apporté à chaque bras

L'écart A3 − A2 dit si le backbone aide. Cette cellule pose l'autre question, tout aussi
utile au papier : **chaque bras a-t-il progressé par rapport à lui-même ?**

Un alignement qui ne bougerait rien dans les deux bras rendrait l'écart entre bras difficile
à interpréter, même mesuré proprement.

In [ ]:
PROGRESSIONS = [
    ("A2 : base -> SFT",        "A0_base", "A2s_sft"),
    ("A2 : base -> SFT+DPO",    "A0_base", "A2d_sft_dpo"),
    ("A3 : base -> SFT",        "A1_base", "A3s_sft"),
    ("A3 : base -> SFT+DPO",    "A1_base", "A3d_sft_dpo"),
]

for etiquette, avant, apres in PROGRESSIONS:
    if avant not in e4 or apres not in e4:
        print(f"{etiquette:<24} incomplet"); continue
    mc = mcnemar_p(justesse[avant], justesse[apres])
    ecart = e4[apres]["accuracy"] - e4[avant]["accuracy"]
    print(f"{etiquette:<24} {ecart:+.4f}   desaccords {mc['discordant']:>3}   "
          f"p = {mc['p']:.4g}  ->",
          "REEL" if mc["p"] < 0.05 else "indiscernable")

print("\nUn gain nul dans LES DEUX bras rendrait l'ecart entre bras peu interpretable,")
print("meme mesure proprement. C'est le controle que cette cellule apporte.")

In [ ]:
# Colab et Kaggle effacent tout: recopier les chiffres ou recuperer le fichier.
print((SORTIES / "E4_uhura.json").read_text(encoding="utf-8")[:1200])

---

## Suite

**E4 sur les deux autres axes** : `11_E4_afrimgsm.ipynb` pour l'utilité et l'oubli
catastrophique, `12_E4_afrihate.ipynb` pour le transfert inter-axes. Les deux peuvent tourner
en parallèle — Kaggle autorise deux sessions GPU simultanées.

**Rappel de lecture.** L'axe Honest est celui sur lequel on **entraîne** : un gain y est
attendu, et c'est son ampleur *relative entre A2 et A3* qui porte le claim. AfriMGSM et
AfriHate ne sont pas entraînés, et répondent à d'autres questions — oubli catastrophique pour
l'un, transfert inter-axes pour l'autre.